# __Advanced Packages: `rioxarray`__
*Before starting this tutorial on `rioxarray`, switch your kernel to "rioxarray".*

Rioxarray is an extension to the `xarray` package. Xarray is a package that works with multidimensional arrays. Rioxarray expands the possibilities of these multidimensional arrays by adding geospatial components to them making rioxarray a great option for working with satellite imagery and climate data. I plan to use this in my final project, since I will largely be dealing with raster data.

*<span style='color: Gray;'>This lecture was created by Ryan Poulsen on November 11, 2025.</span>*

In [ ]:
import rioxarray as rxr

## __<span style='color: Red;'>Loading and Intro to Raster Data</span>__

The data we will be using in this overview is from the Landsat 9 satellite, downloaded from the USGS EarthExplorer. The area of interest is over [Bastrop, Texas](https://www.google.com/maps/place/Bastrop,+TX+78602/@30.1096731,-97.5862841,125953m/data=!3m1!1e3!4m6!3m5!1s0x86449a5be8d1db69:0x340d3c665badd960!8m2!3d30.1104947!4d-97.3152701!16zL20vMF94dGw?entry=ttu&g_ep=EgoyMDI1MTExMS4wIKXMDSoASAFQAw%3D%3D), a city affectionately known for its [Lost Pines](https://en.wikipedia.org/wiki/Lost_Pines_Forest).

If you look in `~/ASC_students/ryan/advanced_packages/data/`, you'll notice that there are four files. Each of these is a different band of the same image, meaning that each one represents a different portion of the electromagnetic spectrum (red, green, blue, etc). You can use different combinations of bands to create true-color images, or false-color images for different analyses, or even add and subtract bands from eachother to calculate different indices. Read about [bands and landsat 9 here](https://www.usgs.gov/faqs/what-are-band-designations-landsat-satellites).

To start, let's just load band 2, or the Blue band. To load raster data using rioxarray, we use `rxr.open_rasterio()`.

In [ ]:
import os

bastrop_data_src = os.path.expanduser("~/ASC_students/ryan/advanced_packages/data/LC09_L2SP_026039_20251105_20251106_02_T1_SR_B2.TIF")

bastrop_data = rxr.open_rasterio(filename=bastrop_data_src) # Opens the raster file at the specified path.

Great! Now we've got our data loaded. Let's learn a little bit about our data.

In [ ]:
# What data type is our data stored as?
type(bastrop_data)

When you run the above code, you'll notice that the data is stored as `xarray.DataArray`. Our raster image is just an array of integers, with each integer corresponding to the amount of blue light at a specific pixel.

In [ ]:
# What is the shape of our data?
bastrop_data.shape

In [ ]:
# What are the coordinates of our data?
bastrop_data.coords

By getting the shape of the data, you'll see that there is only one band and the dimensions of the image are 7681 by 7551 pixels.

In [ ]:
# What is the projection of our data?
print(bastrop_data_b2.rio.crs)

`bastrop_data.rio.crs` printed "EPSG:32614". The EPSG code is a unique code dedicated to specific map projections. This map projection is the [WGS 84 / UTM zone 14N](https://epsg.io/32614) projection. The projection also tells us what units we're working with. This data is in meters, which explains why our coordinates are extremely large numbers.

Unfortunately, there isn't much we can do with just one band of a satellite image. The following code snippet uses the same `rxr.open_rasterio()` function, but in a more advanced way to allowing us to more easily analyze and manipulate the data.

In [ ]:
import os
import xarray as xr

bastrop_data_src = os.path.expanduser("~/ASC_students/ryan/advanced_packages/data/")

band_files = [f"{bastrop_data_src}LC09_L2SP_026039_20251105_20251106_02_T1_SR_B2.TIF", f"{bastrop_data_src}LC09_L2SP_026039_20251105_20251106_02_T1_SR_B3.TIF", f"{bastrop_data_src}LC09_L2SP_026039_20251105_20251106_02_T1_SR_B4.TIF", f"{bastrop_data_src}LC09_L2SP_026039_20251105_20251106_02_T1_SR_B5.TIF"]
band_names = ['B2', 'B3', 'B4', 'B5'] # B2 is blue, B3 is green, B4 is red, and B5 is Near-Infrared (NIR)

das = [rxr.open_rasterio(band_file, masked=True).squeeze("band", drop=True) for band_file in band_files] # Open the rasters! .squeeze() simplifies the data and makes it simpler to stack.
das = [d.rio.reproject_match(das[0]) for d in das] # reproject_match() ensures all rasters are in the same projection

bastrop_data = xr.Dataset({n: d for n, d in zip(band_names, das)}).to_array(dim="band") # Creates a new DataArray with each band stacked.

Great, now we've got all the data loaded and stacked! Let's look at the shape of our data once again to see if anything changed.

In [ ]:
bastrop_data.shape

Look! We now have four bands. Let's go do some basic band math using this data!

## __<span style='color: Green;'>Raster Data Analysis: Basic Band Math</span>__

Let's try some basic manipulation of raster data: calculating a [band/spectral index](https://www.geo.university/pages/spectral-indices-with-multispectral-satellite-data). For this tutorial, let's calculate the Normalized Difference Vegetation Index (NDVI). This index is able to inform us about vegetation health by exploiting the reflective characteristics of vegetation. You can learn more about it [here](https://gisgeography.com/ndvi-normalized-difference-vegetation-index/). NDVI is calculated using the following formula: *NDVI = (NIR - Red)/(NIR + Red)*

*With that in mind, let's do it!*

In [ ]:
# First, we need to select the Red and Near-Infrared (NIR) bands.
red = bastrop_data.sel(band="B4")
nir = bastrop_data.sel(band="B5")

Both our `red` and `nir` variables are individual `xarray.DataArray`s. DataArrays are great because we can perform basic operations on all pixels at once, like subtracting or adding one band to another. This is useful in the calculation of spectral indices.

In [ ]:
# Now let's calculate the index.
ndvi = (nir - red) / (nir + red)

# By definition, NDVI ranges from -1 to 1. So let's clip the values accordingly.
ndvi = ndvi.clip(min=-1, max=1)

# Let's also set the name of the newly calculated index to "NDVI".
ndvi.attrs["long_name"] = "NDVI"

It's that easy! But... we don't really know what our original data or this index looks like. Let's fix that.

## __<span style='color: Blue;'>Visualizing Raster Data with `matplotlib`</span>__

The great thing is, `rioxarray` works pretty seamlessly with `matplotlib`, so mapping is a breeze!

In [ ]:
import matplotlib.pyplot as plt

First, let's reproject our data to a different projection that's a little easier to work with (so we can use decimal degrees instead of meters as our coordinates!).

Now, let's load in the true-color data. A true-color image is RGB, so we're going to select bands 1, 2, and 3.

In [ ]:
plt.figure(figsize=(8, 8))

# Use the sel() method to select the bands we want to display.
bastrop_data.sel(band=['B4', 'B3', 'B2']).plot.imshow(vmin=0, vmax=20000)

# Add title and axis labels.
plt.title("LANDSAT-9 Imagery over Bastrop, TX")
plt.xlabel("x")
plt.ylabel("y")

plt.show()

Look at that! We've got imagery! It's a rather large area, and we're ideally going to focus on a portion of Bastrop, not the western side of Houston. Let's reproject our data (so we have easier units, decimal degrees) and clip our data to cover a smaller area.

In [ ]:
bastrop_data_proj = bastrop_data.rio.reproject("EPSG:4326") # reproject to WGS84

bastrop_data_proj_clip = bastrop_data_proj.rio.clip_box(minx=-97.1672, miny=30.0531, maxx=-97.1142, maxy=30.1117) # clip raster to aoi

Let's also check the shape of our data, to see if it changed.

In [ ]:
bastrop_data_proj_clip.shape

Nice! It's a whole lot smaller now. Let's use matplotlib to visualize it again.

In [ ]:
plt.figure(figsize=(8, 8))

# Use the sel() method to select the bands we want to display.
bastrop_data_proj_clip.sel(band=['B4', 'B3', 'B2']).plot.imshow(vmin=0, vmax=20000)

# Add title and axis labels.
plt.title("LANDSAT-9 Imagery over Bastrop, TX")
plt.xlabel("x")
plt.ylabel("y")

plt.show()

Woah! Wait... are those... _words?_

I mean, everything is bigger in Texas after all. "Luecke" is the last name of Jimmie Luecke, the man who decided to carve his name into a 3-mile strip of his land because, well, he could. This Texas-sized signature is even used by NASA to assess the spatial resolution of photographs taken by astronauts! Read the article here: [This Texan’s 3-Mile-Long Signature in Trees Can Be Seen From Space](https://texashillcountry.com/signature-trees-seen-space/)

Now, let's reproject, clip, and visualize the NDVI data to assess the health of the *[Pinus taeda](https://en.wikipedia.org/wiki/Pinus_taeda)* trees in Luecke's signature.

In [ ]:
ndvi_proj = ndvi.rio.reproject("EPSG:4326") # reproject to WGS84

ndvi_proj_clip = ndvi_proj.rio.clip_box(minx=-97.1672, miny=30.0531, maxx=-97.1142, maxy=30.1117) # clip raster to aoi

In [ ]:
plt.figure(figsize=(8, 8))

# Since we only have one band, NDVI, we don't need to use .sel().
ndvi_proj_clip.plot.imshow(vmin=-(ndvi_proj_clip.max()), vmax=ndvi_proj_clip.max(), cmap="RdYlGn")

# Add title and axis labels.
plt.title("Luecke Property NDVI")
plt.xlabel("x")
plt.ylabel("y")

plt.show()

Look at that NDVI image! It looks quite different from our true-color image. Notice how the "C" in "LUECKE" looks like an "L"? That's because in recent years some of Luecke's pines have died. We're able to see them in the true-color image, but since NDVI is an estimate of vegetation health, the dead trees show up with a much lower NDVI value than the healthy stands of forest.

Now you can move on to the Knowledge Check for this lecture. But first, credit where credit is due, I used this [resource](https://geog-312.gishub.org/book/geospatial/rioxarray.html) from the University of Tennessee, Knoxville by Qiusheng Wu for my rioxarray knowledge. There is so much more to rioxarray, so I *highly recommend* checking it out if you want to learn more. Dr. Spera was also a huge help with data troubleshooting 

## __`rioxarray` Knowledge Check__

Our original data was in the file format of .tif. These files, GeoTiffs, are very useful as they are images that retain geographic attributes, like projections and coordinates. We did a lot of manipulation with our data, but we would also like to be able to save our manipulated data as a GeoTiff so we have it for later, or so we can bring it into other programs like ArcGIS Pro.

In this knowledge check you will:
1. __Clip__ the data to a smaller area of interest (AOI), someplace other than the Luecke property! 
    * Where did you choose?
    * What type of land covers are there?
3. __Display__ your AOI using a different, [false-color](https://earthobservatory.nasa.gov/features/FalseColor), band combination using B5 as one of the selected bands.
    * Why did you choose this combination?
    * Does it show you anything you couldn't see before in the true-color image?
2. __Calculate__ a [Normalized Difference Water Index (NDWI)](https://geoai.au/ndwi-normalized-difference-water-index-with-remote-sensing/) for your clipped raster.
3. __Display__ the NDWI image using a different [color map (cmap)](https://matplotlib.org/stable/users/explain/colors/colormaps.html) than the NDVI image.
    * What cmap did you choose, and why?
4. __[Export](https://corteva.github.io/rioxarray/stable/examples/convert_to_raster.html)__ the NDWI image as a GeoTiff so that it can be used later.

The code to load the data has been included below for convenience.

In [ ]:
import rioxarray as rxr
import os
import xarray as xr

data_src = os.path.expanduser("~/ASC_students/ryan/advanced_packages/data/")

band_files = [f"{data_src}LC09_L2SP_026039_20251105_20251106_02_T1_SR_B2.TIF", f"{data_src}LC09_L2SP_026039_20251105_20251106_02_T1_SR_B3.TIF", f"{data_src}LC09_L2SP_026039_20251105_20251106_02_T1_SR_B4.TIF", f"{data_src}LC09_L2SP_026039_20251105_20251106_02_T1_SR_B5.TIF"]
band_names = ['B2', 'B3', 'B4', 'B5'] # B2 is blue, B3 is green, B4 is red, and B5 is Near-Infrared (NIR)

das = [rxr.open_rasterio(band_file, masked=True).squeeze("band", drop=True) for band_file in band_files] # Open the rasters! .squeeze() simplifies the data and makes it simpler to stack.
das = [d.rio.reproject_match(das[0]) for d in das] # reproject_match() ensures all rasters are in the same projection

data = xr.Dataset({n: d for n, d in zip(band_names, das)}).to_array(dim="band") # Creates a new DataArray with each band stacked.